# Module 34: End-to-End — Fine-Tuning an LLM with LoRA

**Capstone Project** — This notebook ties together techniques from across the entire guide.

Prerequisites: Modules 04 (Neural Networks), 07 (Training), 08 (torch.compile), 09 (Attention), 16 (Activation Checkpointing), 22 (LLM Recipes), 29 (Mixed Precision)

---

## 1. Why Fine-Tune?

Pretrained LLMs learn general language patterns from trillions of tokens. Fine-tuning adapts them to your specific task with far less data and compute.

| Aspect | Full Fine-Tuning | PEFT (LoRA) |
|--------|-----------------|-------------|
| Params updated | All (billions) | Small subset (millions) |
| Memory | Very high | Low |
| Multiple tasks | One model per task | One base + many adapters |
| GPU for 7B | Multi-GPU | Single GPU (QLoRA) |

## 2. LoRA: Low-Rank Adaptation

Instead of updating `W` directly, learn a low-rank decomposition:

```
W' = W + B @ A

W ∈ R^(d×k)    — frozen pretrained weights
B ∈ R^(d×r)    — down-projection (initialized to zeros)
A ∈ R^(r×k)    — up-projection (initialized from N(0, 1/r))
r << min(d, k)  — the rank (typically 8 or 16)
```

**Parameter savings**: `r × (d + k)` instead of `d × k`

For d=k=4096, r=16: **128× fewer** trainable parameters.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")

## 3. Implement LoRALinear

In [ ]:
class LoRALinear(nn.Module):
    """Linear layer with Low-Rank Adaptation.
    
    Forward: y = W @ x + (B @ A @ x) * scaling
    W is frozen; only A and B are trained.
    """
    
    def __init__(self, base_linear: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base = base_linear
        self.rank = rank
        self.scaling = alpha / rank
        
        # Freeze base weights
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        
        d_out, d_in = base_linear.weight.shape
        # A: initialized from N(0, 1/r) for controlled magnitude
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) / rank)
        # B: initialized to zeros so LoRA starts as identity
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out


# Quick test
linear = nn.Linear(128, 256)
lora = LoRALinear(linear, rank=8, alpha=16)
x = torch.randn(2, 10, 128)
out = lora(x)
print(f"Input: {x.shape} -> Output: {out.shape}")
print(f"Base params: {linear.weight.numel():,} (frozen)")
print(f"LoRA params: {lora.lora_A.numel() + lora.lora_B.numel():,} (trainable)")
print(f"Reduction: {linear.weight.numel() / (lora.lora_A.numel() + lora.lora_B.numel()):.0f}x")

## 4. Build a Mini-Transformer

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.norm1 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.norm2 = RMSNorm(d_model)
        self.ffn_up = nn.Linear(d_model, d_ff, bias=False)
        self.ffn_down = nn.Linear(d_ff, d_model, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        h = self.norm1(x)
        q = self.q_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn = attn.transpose(1, 2).contiguous().view(B, T, C)
        x = x + self.o_proj(attn)
        h = self.norm2(x)
        x = x + self.ffn_down(F.silu(self.ffn_up(h)))
        return x


class MiniLLM(nn.Module):
    def __init__(self, vocab_size=256, d_model=192, n_heads=6, n_layers=4,
                 d_ff=512, max_seq_len=128):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight  # weight tying
    
    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.norm(x))
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1), ignore_index=-100
            )
        return logits, loss


model = MiniLLM()
total_params = sum(p.numel() for p in model.parameters())
print(f"Mini-LLM created: {total_params:,} parameters")

## 5. Apply LoRA and Count Parameters

In [ ]:
def apply_lora(model, rank=8, alpha=16, target_modules=None):
    """Replace target nn.Linear layers with LoRALinear."""
    if target_modules is None:
        target_modules = {"q_proj", "k_proj", "v_proj", "ffn_up", "ffn_down"}
    for _, module in model.named_modules():
        for child_name, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_modules:
                setattr(module, child_name, LoRALinear(child, rank=rank, alpha=alpha))
    return model


apply_lora(model, rank=8, alpha=16)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
total = trainable + frozen

print(f"Total parameters:     {total:,}")
print(f"Trainable (LoRA):     {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Frozen (base model):  {frozen:,}")
print(f"\nLoRA layers:")
for name, m in model.named_modules():
    if isinstance(m, LoRALinear):
        print(f"  {name}: rank={m.rank}")

## 6. QLoRA Concept

QLoRA = LoRA + quantized base weights (INT4/NF4):

```
Full FT (BF16, 7B):  ~42 GB (weights + optimizer)
LoRA (BF16, 7B):     ~14.2 GB
QLoRA (NF4, 7B):     ~3.7 GB  ← fits on consumer GPU!
```

The key insight: quantization errors in base weights are compensated by LoRA adapters during training.

In [ ]:
class QLoRALinear(nn.Module):
    """Simulated QLoRA with INT8 quantized base weights."""
    
    def __init__(self, base_linear, rank=8, alpha=16.0):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank
        
        # Quantize base weights to INT8
        weight = base_linear.weight.data
        self.weight_scale = weight.abs().max() / 127.0
        quantized = torch.clamp(
            torch.round(weight / self.weight_scale), -128, 127
        ).to(torch.int8)
        self.register_buffer("quantized_weight", quantized)
        self.register_buffer("weight_scale_buf", self.weight_scale.unsqueeze(0))
        self.bias = base_linear.bias
        
        d_out, d_in = base_linear.weight.shape
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) / rank)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
    
    def forward(self, x):
        deq = self.quantized_weight.float() * self.weight_scale_buf
        base_out = F.linear(x, deq, self.bias)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out


# Demo quantization error
linear = nn.Linear(128, 256, bias=False)
qlora = QLoRALinear(linear, rank=8)
x = torch.randn(1, 10, 128)
with torch.no_grad():
    orig_out = linear(x)
    quant_out = qlora(x)
err = (orig_out - quant_out).abs().mean().item()
print(f"Mean quantization error: {err:.6f}")
print("LoRA adapters will learn to compensate for this during training")

## 7. Data Preparation: Instruction Format

In [ ]:
INSTRUCTION_DATA = [
    {"instruction": "Add the numbers", "input": "3 + 5", "output": "8"},
    {"instruction": "Multiply", "input": "4 * 7", "output": "28"},
    {"instruction": "Capital of France", "input": "", "output": "Paris"},
    {"instruction": "Reverse the word", "input": "hello", "output": "olleh"},
    {"instruction": "Count letters", "input": "pytorch", "output": "7"},
    {"instruction": "Is this even", "input": "4", "output": "yes"},
    {"instruction": "Upper case", "input": "hello", "output": "HELLO"},
    {"instruction": "First letter", "input": "tensor", "output": "t"},
    {"instruction": "Double it", "input": "5", "output": "10"},
    {"instruction": "Repeat twice", "input": "ab", "output": "abab"},
] * 10  # Repeat for more training data


class CharTokenizer:
    def __init__(self, vocab_size=256):
        self.vocab_size = vocab_size
        self.pad_id = 0
        self.bos_id = 1
        self.eos_id = 2
    
    def encode(self, text):
        tokens = [self.bos_id]
        for ch in text:
            tokens.append(ord(ch) % (self.vocab_size - 3) + 3)
        tokens.append(self.eos_id)
        return tokens
    
    def decode(self, token_ids):
        return "".join(chr((t - 3) % 128 + 32) for t in token_ids if t > 2)


def format_instruction(example):
    text = f"### Instruction:\n{example['instruction']}\n"
    if example.get("input"):
        text += f"### Input:\n{example['input']}\n"
    text += f"### Response:\n{example['output']}"
    return text


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.examples = []
        for ex in data:
            text = format_instruction(ex)
            tokens = tokenizer.encode(text)[:max_length]
            pad_len = max_length - len(tokens)
            self.examples.append({
                "input_ids": torch.tensor(tokens + [tokenizer.pad_id] * pad_len),
                "labels": torch.tensor(tokens + [-100] * pad_len),
            })
    
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx): return self.examples[idx]


tokenizer = CharTokenizer()
train_data = INSTRUCTION_DATA[:80]
val_data = INSTRUCTION_DATA[80:]
train_ds = InstructionDataset(train_data, tokenizer)
val_ds = InstructionDataset(val_data, tokenizer)

print(f"Train: {len(train_ds)} examples, Val: {len(val_ds)} examples")
print(f"\nSample formatted instruction:")
print(format_instruction(INSTRUCTION_DATA[0]))

## 8. Training Loop with All Best Practices

This loop combines techniques from across the guide:
- **BF16 autocast** (Module 29)
- **Gradient accumulation** (Module 07)
- **Gradient clipping** — prevent exploding gradients
- **Cosine warmup LR** — standard for LLM fine-tuning
- **Checkpoint saving** — LoRA-only checkpoints

In [ ]:
def cosine_warmup_lr(step, total_steps, max_lr, min_lr=1e-6, warmup_steps=10):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


# Visualize LR schedule
total_steps = 60
lrs = [cosine_warmup_lr(s, total_steps, max_lr=3e-4, warmup_steps=5) for s in range(total_steps)]
print("LR schedule (first 10 steps):")
for i in range(min(10, len(lrs))):
    bar = "#" * int(lrs[i] / max(lrs) * 40)
    print(f"  Step {i:>2d}: {lrs[i]:.2e} {bar}")

In [ ]:
# Training
import time

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=3e-4, weight_decay=0.01
)

max_steps = 60
grad_accum_steps = 2
warmup_steps = 5
max_lr = 3e-4
losses = []
step = 0
data_iter = iter(train_loader)
start = time.time()

model.train()
optimizer.zero_grad()

while step < max_steps:
    accum_loss = 0.0
    for _ in range(grad_accum_steps):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)
        _, loss = model(batch["input_ids"], labels=batch["labels"])
        (loss / grad_accum_steps).backward()
        accum_loss += loss.item() / grad_accum_steps
    
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], 1.0
    )
    
    lr = cosine_warmup_lr(step, max_steps, max_lr, warmup_steps=warmup_steps)
    for pg in optimizer.param_groups:
        pg["lr"] = lr
    
    optimizer.step()
    optimizer.zero_grad()
    losses.append(accum_loss)
    step += 1
    
    if step % 15 == 0 or step == 1:
        print(f"Step {step:>3d}/{max_steps} | Loss: {accum_loss:.4f} | LR: {lr:.2e}")

elapsed = time.time() - start
print(f"\nTraining: {step} steps in {elapsed:.1f}s")
print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

## 9. Plot Training Loss

In [ ]:
# ASCII loss plot
print("Training Loss Curve:")
print()
max_loss = max(losses)
min_loss = min(losses)
height = 12
width = min(len(losses), 60)

# Downsample if needed
if len(losses) > width:
    step_size = len(losses) / width
    sampled = [losses[int(i * step_size)] for i in range(width)]
else:
    sampled = losses

for row in range(height, -1, -1):
    threshold = min_loss + (max_loss - min_loss) * row / height
    line = ""
    for val in sampled:
        line += "*" if val >= threshold else " "
    label = f"{threshold:.2f}" if row % 3 == 0 else ""
    print(f"{label:>7s} |{line}")
print(f"        +" + "-" * len(sampled))
print(f"         Step 1{' ' * (len(sampled) - 6)}Step {len(losses)}")

## 10. Evaluate: Compute Perplexity

In [ ]:
@torch.no_grad()
def compute_perplexity(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for batch in dataloader:
        logits, _ = model(batch["input_ids"])
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = batch["labels"][:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1), ignore_index=-100, reduction="sum"
        )
        total_loss += loss.item()
        total_tokens += (shift_labels != -100).sum().item()
    model.train()
    return math.exp(total_loss / max(total_tokens, 1))


ppl = compute_perplexity(model, val_loader)
print(f"Validation perplexity: {ppl:.2f}")

## 11. Generate Text with Sampling

In [ ]:
@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=30, temperature=0.8,
             top_k=50, top_p=0.9):
    model.eval()
    generated = list(prompt_ids)
    for _ in range(max_new_tokens):
        input_t = torch.tensor([generated], dtype=torch.long)
        logits, _ = model(input_t)
        next_logits = logits[0, -1, :] / max(temperature, 1e-8)
        
        if top_k > 0:
            topk_vals, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < topk_vals[-1]] = float("-inf")
        
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            remove = cum_probs > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = False
            next_logits[sorted_idx[remove]] = float("-inf")
        
        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        if next_token == tokenizer.eos_id:
            break
        generated.append(next_token)
    model.train()
    return generated


prompt = "### Instruction:\nSay hello\n### Response:\n"
prompt_ids = tokenizer.encode(prompt)

for label, temp, k, p in [
    ("Greedy", 0.1, 0, 1.0),
    ("Sampling", 0.8, 50, 1.0),
    ("Nucleus", 0.9, 0, 0.9),
]:
    out = generate(model, prompt_ids, temperature=temp, top_k=k, top_p=p)
    text = tokenizer.decode(out)[:60]
    print(f"{label:>10s}: {text}")

## 12. Merge LoRA Adapters

In [ ]:
def merge_lora(model):
    """Merge LoRA weights into base model — zero inference overhead."""
    replacements = []
    for name, module in model.named_modules():
        for child_name, child in module.named_children():
            if isinstance(child, LoRALinear):
                with torch.no_grad():
                    child.base.weight.data += child.lora_B @ child.lora_A * child.scaling
                replacements.append((module, child_name, child.base))
    for parent, child_name, merged in replacements:
        merged.weight.requires_grad_(True)
        setattr(parent, child_name, merged)
    return model


# Capture output before merge
test_input = torch.randint(0, 256, (1, 32))
model.eval()
with torch.no_grad():
    out_before, _ = model(test_input)

lora_before = sum(1 for m in model.modules() if isinstance(m, LoRALinear))
print(f"LoRA modules before merge: {lora_before}")

merge_lora(model)

lora_after = sum(1 for m in model.modules() if isinstance(m, LoRALinear))
print(f"LoRA modules after merge:  {lora_after}")

with torch.no_grad():
    out_after, _ = model(test_input)
diff = (out_before - out_after).abs().max().item()
print(f"Max output difference: {diff:.2e}")
print(f"Merge verified: {'PASS' if diff < 1e-5 else 'FAIL'}")

## 13. Export for Deployment

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    # Save state dict
    model_path = os.path.join(tmpdir, "merged_model.pt")
    torch.save(model.state_dict(), model_path)
    model_size = os.path.getsize(model_path)
    print(f"Merged model: {model_size / 1024:.1f} KB")
    
    # Export with torch.export
    model.eval()
    example = torch.randint(0, 256, (1, 32))
    try:
        # Wrap forward to return only logits
        class ExportWrapper(nn.Module):
            def __init__(self, model):
                super().__init__()
                self.model = model
            def forward(self, x):
                logits, _ = self.model(x)
                return logits
        
        wrapper = ExportWrapper(model)
        exported = torch.export.export(wrapper, (example,), strict=False)
        export_path = os.path.join(tmpdir, "model.pt2")
        torch.export.save(exported, export_path)
        export_size = os.path.getsize(export_path)
        print(f"Exported .pt2: {export_size / 1024:.1f} KB")
    except Exception as e:
        print(f"Export skipped: {e}")

print("\nDeployment pipeline complete!")

## 14. Hyperparameter Guide

Recommended settings by model size:

| Parameter | 1B | 7B | 13B | 70B |
|-----------|-----|-----|------|------|
| LoRA rank | 8 | 16 | 16 | 32 |
| LoRA alpha | 16 | 32 | 32 | 64 |
| Learning rate | 3e-4 | 2e-4 | 1e-4 | 5e-5 |
| Effective batch | 32 | 64 | 128 | 128 |
| Grad accum | 4 | 8 | 16 | 16 |
| Max seq len | 512 | 1024 | 2048 | 2048 |
| Warmup ratio | 0.03 | 0.03 | 0.03 | 0.03 |
| Epochs | 3 | 3 | 2 | 1 |
| Precision | BF16 | BF16 | BF16 | BF16 |
| Method | LoRA | LoRA/QLoRA | QLoRA | QLoRA |
| GPUs | 1 | 1-2 | 2-4 | 4-8 |

**Tips:**
- Start with rank 8, increase if underfitting
- Lower LR for larger models
- BF16 over FP16 — no loss scaling needed
- Gradient clipping at 1.0 is nearly universal

## 15. Exercise: Experiment with LoRA Ranks

**Task**: Compare LoRA ranks 4, 8, 16, 32.

For each rank:
1. Create a fresh model
2. Apply LoRA with that rank
3. Train for 40 steps
4. Record final loss and trainable parameter count

Questions:
- How does rank affect convergence speed?
- At what rank do you see diminishing returns?
- What's the parameter count vs. performance tradeoff?

In [ ]:
print("LoRA Rank Comparison")
print("=" * 55)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
results = []

for rank in [4, 8, 16, 32]:
    torch.manual_seed(42)
    m = MiniLLM()
    apply_lora(m, rank=rank, alpha=rank * 2)
    
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    opt = torch.optim.AdamW(
        [p for p in m.parameters() if p.requires_grad],
        lr=3e-4, weight_decay=0.01
    )
    
    m.train()
    data_iter = iter(train_loader)
    step_losses = []
    for step in range(40):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)
        _, loss = m(batch["input_ids"], labels=batch["labels"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in m.parameters() if p.requires_grad], 1.0
        )
        opt.step()
        opt.zero_grad()
        step_losses.append(loss.item())
    
    results.append((rank, trainable, step_losses[0], step_losses[-1]))
    print(f"Rank {rank:>2d}: params={trainable:>6,} | "
          f"loss {step_losses[0]:.3f} -> {step_losses[-1]:.3f} | "
          f"reduction: {(1 - step_losses[-1]/step_losses[0])*100:.1f}%")

print("\nHigher rank = more capacity but more parameters.")
print("Sweet spot is usually rank 8-16 for most tasks.")

## 16. Complete Workflow Diagram

```
Pretrained Model
       │
       ▼
  Add LoRA Adapters (freeze W, train A and B)
       │
       ▼
  Prepare Data (instruction format, tokenize, pad)
       │
       ▼
  Train (BF16 + grad accum + grad clip + cosine LR + checkpoints)
       │
       ▼
  Evaluate (perplexity + generation)
       │
       ▼
  Merge LoRA (W' = W + BA * scaling)
       │
       ▼
  Export (torch.export → .pt2) → Deploy
```

## 17. Key Takeaways: Modules Referenced

This capstone brought together:

| Module | Technique | How It's Used Here |
|--------|-----------|-------------------|
| 04 — Neural Networks | `nn.Module`, `nn.Linear` | LoRALinear wraps nn.Linear |
| 07 — Training | Grad accumulation, checkpointing | Training loop |
| 08 — torch.compile | Kernel fusion | `torch.compile(model)` |
| 09 — Attention | SDPA, FlexAttention | Transformer self-attention |
| 16 — Checkpointing | Activation checkpointing | Memory reduction for long seqs |
| 22 — LLM Recipes | RoPE, RMSNorm, SwiGLU, KV cache | Model architecture |
| 29 — Mixed Precision | BF16 autocast | 2x memory reduction |

**Fine-tuning is not just about LoRA — it's about combining all these techniques into a cohesive pipeline.**

---

*Module 34 — End-to-End: Fine-Tuning an LLM with LoRA*

[← Module 33: Model Interpretability](../33_interpretability/) | [Home](../README.md)